In [ ]:
# Import necessary libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from parameters_py.config import (
					FOLDER_NAME,FOLDER_OUTPUT,
                    MONTH,)

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (config.py, line 243)

# Text based on:

- Radhakrishnan & Anilkumar. **Inversion for water column sound speed profile from acoustic travel times using empirical orthogonal functions**. J. Acoust. Soc. Am. 1 December 2024; 156 (6): 4061–4072.
- Fortin, F. A., Rainville, F. M., Gardner, M., Parizeau, M., and Gagné, C. **DEAP: Evolutionary Algorithms Made Easy**, Journal of Machine Learning Research, pp. 2171-2175, no 13, jul 2012.

# EOF Analysis of Sound Speed ​​Profiles (SSP) – GLORYS Data

This notebook implements Empirical Orthogonal Function (EOF) analysis on 20 years of SSP data extracted from GLORYS for a specific location (1D spatial + time).
The objective is to identify the dominant modes of temporal and spatial variability in the water column.

# Inputs and outputs

### Directory for saving model and other related stuffs: 

In [2]:
data_path = FOLDER_OUTPUT+FOLDER_NAME+'/DATA/'

### Directory for saving figures: 

In [3]:
figures_path = FOLDER_OUTPUT+FOLDER_NAME+'/FIGURES/'

---------

# Loading the Glorys data processed previosly 

In [4]:
glorys_filename_feather = data_path+'data_glorys/VELOCITY_DENSITY_KMeans.feather'

glorys_data = pd.read_feather(glorys_filename_feather)

In [5]:
glorys_data

,latitude,longitude,month,date,density_cut,impedance,density,sound_speed,depth,k-means
0,-25.5,-43.0,Jan,1993-01-01 12:00:00,"[1025.203518252913, 1025.3801395315163, 1025.5...","[1574908.9278395972, 1574702.810231142, 157452...","[1024.5866955500674, 1024.6264923723634, 1024....","[1537.1163169301938, 1536.8554511851069, 1536....","[0.49402499198913574, 1.5413750410079956, 2.64...",3
0,-25.5,-43.0,Jan,1993-01-02 12:00:00,"[1025.209588278287, 1025.386587140039, 1025.53...","[1575888.0834565682, 1575478.0369867343, 15752...","[1024.4714722258998, 1024.5441486499249, 1024....","[1538.2449645304316, 1537.73562521712, 1537.41...","[0.49402499198913574, 1.5413750410079956, 2.64...",3
0,-25.5,-43.0,Jan,1993-01-03 12:00:00,"[1025.207373956924, 1025.385125680771, 1025.54...","[1575493.2311382303, 1575514.3672327222, 15754...","[1024.5330671978652, 1024.5382101937412, 1024....","[1537.7670878376439, 1537.779998400246, 1537.7...","[0.49402499198913574, 1.5413750410079956, 2.64...",3
0,-25.5,-43.0,Jan,1993-01-04 12:00:00,"[1025.191801261887, 1025.37542397262, 1025.547...","[1575331.682373201, 1575354.199242716, 1575356...","[1024.6022567644445, 1024.6071714369457, 1024....","[1537.505575429714, 1537.520176667691, 1537.51...","[0.49402499198913574, 1.5413750410079956, 2.64...",3
0,-25.5,-43.0,Jan,1993-01-05 12:00:00,"[1025.1804083763423, 1025.365123874524, 1025.5...","[1575237.121292404, 1575259.6306669232, 157528...","[1024.6278366148217, 1024.6327513885717, 1024....","[1537.3749033568051, 1537.3894973903066, 1537....","[0.49402499198913574, 1.5413750410079956, 2.64...",3
...,...,...,...,...,...,...,...,...,...,...
0,-25.5,-43.0,Dec,2024-12-27 00:00:00,"[1025.291580233775, 1025.4245706277068, 1025.5...","[1575456.2319856607, 1575483.608715153, 157550...","[1024.6145869185725, 1024.620426052087, 1024.6...","[1537.6086306985833, 1537.626587033375, 1537.6...","[0.49402499198913574, 1.5413750410079956, 2.64...",3
0,-25.5,-43.0,Dec,2024-12-28 00:00:00,"[1025.2292714867058, 1025.3724321500397, 1025....","[1576606.7436333734, 1576531.9732907875, 15764...","[1024.4947616735071, 1024.5195222211196, 1024....","[1538.9114738449166, 1538.8013006066744, 1538....","[0.49402499198913574, 1.5413750410079956, 2.64...",3
0,-25.5,-43.0,Dec,2024-12-29 00:00:00,"[1025.204776477509, 1025.3500971067251, 1025.5...","[1576836.6576271448, 1576820.283249775, 157680...","[1024.506193714861, 1024.5159703801276, 1024.5...","[1539.1187162173542, 1539.0880462944128, 1539....","[0.49402499198913574, 1.5413750410079956, 2.64...",3
0,-25.5,-43.0,Dec,2024-12-30 00:00:00,"[1025.2014520518317, 1025.3529673960697, 1025....","[1576688.0213653257, 1576709.2410880185, 15767...","[1024.5157158259408, 1024.5208597210647, 1024....","[1538.9593317211697, 1538.9723167933275, 1538....","[0.49402499198913574, 1.5413750410079956, 2.64...",3


## Over a profile, we have 9 shots and 9 dispersion curves (receptors):

In [ ]:
obs_data